# 2 How do you know it is any good?

You will spend the rest of your career being handed model output and asked whether it is
good. For everything before this lesson you had an answer: a held-out split and a metric.
For generated text most people quietly stop doing that and start *looking at a few
examples* instead.

Eyeballing is not a metric. It does not scale past about ten items, it has no error bar,
and it silently selects for outputs that are easy to read rather than correct.

The alternative is not complicated, and this notebook builds it in order of cost:

1. compare against ground truth, where you have it
2. use a model to judge, where you do not
3. **and then judge the judge** — the step that almost never happens

Everything here runs on CPU in seconds, because the expensive part already ran. The
judgements were generated once by `build_judgements.py` against a local model and saved to
`data/judgements.json`. Regenerating them needs ollama and about four minutes; analysing
them needs neither.

Read alongside `references/papers.md` → Zheng et al.,
[arXiv:2306.05685](https://arxiv.org/abs/2306.05685).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

d = json.loads((Path("data") / "judgements.json").read_text())
direct, pairwise = d["direct"], d["pairwise"]

print(f"model            : {d['model']}")
print(f"direct judgements: {len(direct)}")
print(f"pairwise (x2)    : {len(pairwise)} pairs, each asked twice with the order swapped")
print(f"reviews truncated at {d['max_chars']} characters")

The task is IMDB sentiment, chosen because **it has ground truth**. That is the only way
to check a judge at all: you need a set of questions where you already know the answer, or
you are just comparing one opinion with another.

Note the truncation. Reviews are cut at 1200 characters, so the judge often sees part of a
review rather than all of it — the context-window budget from lesson 4, showing up as an
experimental caveat rather than as a concept.

## 2.1 Where ground truth exists, use it

In [ ]:
parsed = [r for r in direct if r["pred"] is not None]
pred = np.array([r["pred"] for r in parsed])
truth = np.array([r["truth"] for r in parsed])

acc = float((pred == truth).mean())
po = acc
pe = sum((pred == k).mean() * (truth == k).mean() for k in (0, 1))
kappa = (po - pe) / (1 - pe)

print(f"accuracy      : {acc:.4f}")
print(f"chance level  : {pe:.4f}   (what you would get by guessing with these marginals)")
print(f"Cohen's kappa : {kappa:.4f}")

**Report kappa, not accuracy.** Accuracy on a balanced two-class task starts at 0.5 for
free; kappa subtracts off what agreement-by-chance would have given you and rescales, so
0 means "no better than guessing" and 1 means perfect. On a skewed task the difference is
dramatic — this is the same trap notebook 03 of lesson 6 hits from the decision side,
where a 2% base rate makes 0.978 accuracy worthless.

Here the model is genuinely strong: it agrees with the human labels about as well as you
could hope for on a task like this.

## 2.2 The output you cannot parse is a result too

In [ ]:
failed = [r for r in direct if r["pred"] is None]
print(f"unparseable: {len(failed)} of {len(direct)}  ({len(failed) / len(direct):.1%})")
for r in failed[:3]:
    print(f"   truth={r['truth']}  raw={r['raw']!r}")

A small percentage, and look at *what* it is: the model answered **"Mixed"**. It did not
malfunction — it declined the binary you imposed, on reviews that genuinely are mixed. The
failure is in the question, not the answer.

That is the part of an LLM pipeline that consumes your afternoon, and it never shows up in
a benchmark score.

**Decide what these count as before you see them**, because there are three defensible
choices and they give different headline numbers: drop them (and report a coverage figure
alongside your accuracy), count them as wrong, or retry with a stricter prompt. Silently
dropping them and reporting accuracy on the remainder is the one that is *not* defensible,
and it is the default behaviour of almost every quick evaluation script.

## 2.3 Now judge the judge

Direct scoring needs ground truth. The reason LLM-as-judge exists is for the case where you
have none: you show the model two candidate outputs and ask which is better.

That is fine, as long as the answer depends on the *candidates* and not on the *order you
listed them in*. Which is testable — ask every pair twice, swapping the positions.

In [ ]:
ok = [p for p in pairwise if p["pos_first"] and p["neg_first"]]

# when the positive review is shown as A, the correct answer is "A";
# when it is shown as B, the correct answer is "B".
acc_first = float(np.mean([p["pos_first"] == "A" for p in ok]))
acc_second = float(np.mean([p["neg_first"] == "B" for p in ok]))
consistent = float(np.mean([(p["pos_first"] == "A") == (p["neg_first"] == "B") for p in ok]))
share_A = float(np.mean([p["pos_first"] == "A" for p in ok] + [p["neg_first"] == "A" for p in ok]))

print(f"pairs usable                          : {len(ok)}")
print(f"accuracy, correct answer listed FIRST : {acc_first:.4f}")
print(f"accuracy, correct answer listed SECOND: {acc_second:.4f}")
print(f"verdict unchanged when swapped        : {consistent:.4f}")
print(f"  -> flip rate                        : {1 - consistent:.4f}")
print(f"share of all answers that were 'A'    : {share_A:.4f}")

**The same pair, the same model, a different answer, depending on which one you typed
first.** Roughly one verdict in twelve flips under a swap, and the model is several points
more accurate when the right answer happens to be listed first.

This is *position bias*, and Zheng et al. document it as one of the standard failure modes
of LLM-as-judge. It matters because of how these evaluations are normally run: you compare
system A against system B, list them in a fixed order, and report a win rate. Some of that
win rate is your ordering.

The fix is cheap and almost nobody does it: **ask both ways and keep only the pairs that
agree**, or average over the two orders. What you cannot do is ask once and report the
number as if the order were not part of the question.

In [ ]:
plt.figure(figsize=(6.5, 3.6))
plt.bar(["correct listed\nfirst", "correct listed\nsecond"], [acc_first, acc_second],
        color=["tab:blue", "tab:orange"])
plt.axhline(0.5, ls="--", color="grey", label="chance")
plt.ylim(0, 1.05); plt.ylabel("accuracy"); plt.legend()
plt.title(f"same judge, same pairs, {len(ok)} comparisons")
plt.tight_layout(); plt.show()

## 2.4 A bias that turns out not to be there

Verbosity bias — judges preferring the longer answer — is the other failure mode everyone
quotes. Test it the obvious way and you get a number that looks alarming, and is not.

In [ ]:
picked_longer = float(np.mean([
    (p["pos_first"] == "A") == (p["n_chars_pos"] > p["n_chars_neg"]) for p in ok]))
base_rate = float(np.mean([p["n_chars_pos"] > p["n_chars_neg"] for p in ok]))

print(f"judge picked the LONGER review in      : {picked_longer:.1%} of pairs")
print(f"the positive review IS the longer one  : {base_rate:.1%} of pairs")

Those two numbers are the same, and that is the whole point.

The judge is right about 93% of the time, so "which one did it pick" is mostly "which one
was positive". If positive reviews in this sample happen to be shorter, then the judge will
appear to prefer shorter text — **and it would appear to prefer longer text on a sample
where the polarity/length correlation ran the other way.** The measurement is reading a
property of the corpus and reporting it as a property of the model.

There is no verbosity bias detectable here. Say so. A confounded metric that produces an
exciting number is worse than no metric, because someone will put it in a slide.

To measure verbosity bias properly you have to break the confound — hold content fixed and
vary only length, which is exercise E3.

## What you now know

- **Kappa, not accuracy**, whenever chance agreement is non-trivial.
- **Unparseable output is a number**, and how you count it must be decided in advance.
- **A judge has to be judged.** Order changes about one verdict in twelve here, and shifts
  accuracy by several points.
- **A bias you can measure is not the same as a bias that is there.** The verbosity result
  was a property of the corpus.
- Everything above ran on a cache. The expensive part happened once; the thinking is free.

The next notebook needs exactly this. A router has to decide whether the cheap model's
answer was good enough — and *that decision is an evaluation*. Without a metric you trust,
there is nothing to route on.

## Exercises

**E1 — how many pairs do you need?** The flip rate is estimated from about a hundred
comparisons. Bootstrap it: resample the pairs with replacement, recompute, and report a
confidence interval. Is the position effect distinguishable from noise, and how many pairs
would you need to halve the interval?

**E2 — does the prompt fix it?** Regenerate a subset with `build_judgements.py` after
adding an instruction such as "the order in which the reviews are presented is arbitrary".
Does the flip rate move? Prompt fixes for position bias are widely recommended and rarely
measured.

**E3 — verbosity, properly.** Break the confound: take one review and make a longer version
by padding it with neutral sentences that add no information. Ask the judge to compare the
two. Now length varies and content does not, so any preference is the judge's.

**E4 — a cheap judge.** Train a classifier on the embedding cache for this task and compare
it with the LLM judge on agreement with ground truth, on cost, and on latency. If the cheap
judge is close, the expensive one needs to justify itself — which is the argument notebook
03 turns into a routing policy.

**E5 — the truncation caveat.** 1200 characters cuts most of these reviews. Regenerate at
400 and at 4000 characters, and plot agreement against the budget. How much of this model's
apparent quality is a function of how much you let it read?

**E6 — self-preference.** Have the model judge its own output against another model's. This
is the bias hardest to correct for, because the two things you are comparing are not
symmetric to the judge. Does it show up here?